# 06 Task 2A: weekly demand forecast

**Purpose:** forecast total and chilled volume (m3) for each depot and brand for ISO weeks 14 to 23 of 2026. These are the weeks around New Year, Vesak and Poson, when reefer and truck capacity is tightest.

**The trap.** Festivals move between ISO weeks every year (notebook 01). Vesak was week 21 in 2024, week 20 in 2025 and is week 18 in 2026. Any method that copies "the same week last year" puts the festival spike in the wrong week.

**Our approach.** Forecast each day, then add the days up into ISO weeks. A daily model learns "the days before a festival are busy and the festival day is closed", so the weekly totals follow the 2026 calendar on their own. Two daily models are blended:
- a readable per-series regression on log volume with a growth trend and calendar effects
- one LightGBM across all series that learns the calendar shape on top of that trend

**Inputs:** clean orders (train and Task 1 test inputs), the calendar and the `not_run` uplift from notebook 03.

**Outputs:** `submissions/submission_task2a.csv`, `data/processed/task2a_forecast.parquet` with P10/P50/P90 ranges, and saved models in `models/`.

In [1]:
import json

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora import forecast as fc
from xora.io import load_processed, save_processed
from xora.paths import MODELS, SUBMISSIONS, raw_file
from xora.plotting import apply_style, SERIES, INK, INK_SOFT

apply_style()
pd.set_option("display.width", 160)

orders = load_processed("orders")
calendar = load_processed("calendar")
weekly_table = load_processed("weekly_demand")

## 1. Daily demand panel

The counting rules match notebook 03's weekly table:
- every order counts once on its order date, including deferred and `not_run` orders
- weeks come from the calendar's `iso_year` and `iso_week`
- weeks 8 to 13 of 2026 come from the Task 1 test inputs, which leave out `not_run` orders, so they get the same small uplift as in notebook 03

Days with no orders are filled with zero, so Style and Tech, which do not order every day, are not overstated. Chilled is forecast only for Fresh; Style and Tech carry no chilled goods.

In [2]:
test_weeks = weekly_table[weekly_table.source == "test inputs"]
uplift = test_weeks.assign(
    factor_total=(test_weeks.total_volume_m3_adj / test_weeks.total_volume_m3).fillna(1.0),
    factor_chilled=(test_weeks.chilled_volume_m3_adj / test_weeks.chilled_volume_m3).fillna(1.0),
)[["depot", "brand", "iso_year", "iso_week", "factor_total", "factor_chilled"]]

panel = fc.daily_panel(orders, calendar, uplift)
weekly = fc.to_weeks(panel, ["y"])

# The daily panel must add back up to notebook 03's weekly table exactly
check = weekly.pivot_table(index=["depot", "brand", "iso_year", "iso_week"], columns="measure", values="y").fillna(0)
ref = weekly_table.set_index(["depot", "brand", "iso_year", "iso_week"])[["total_volume_m3_adj", "chilled_volume_m3_adj"]]
gap = (check[["total", "chilled"]].to_numpy() - ref.loc[check.index].to_numpy())
print(f"largest gap against the weekly table: {np.abs(gap).max():.6f} m3")
panel.groupby(fc.SERIES_KEYS, observed=True).agg(days=("y", "size"), mean_daily_m3=("y", "mean"), last_day=("date", "max"))

largest gap against the weekly table: 0.000000 m3


days  mean_daily_m3   last_day
depot      brand measure                                
Kandy      Fresh chilled   818      26.004836 2026-03-28
                 total     818      71.553423 2026-03-28
           Style total     818      11.584805 2026-03-28
           Tech  total     818       3.035246 2026-03-28
Peliyagoda Fresh chilled   818      50.387904 2026-03-28
                 total     818     136.754166 2026-03-28
           Style total     818      20.263391 2026-03-28
           Tech  total     818       4.258793 2026-03-28